

---


# ADALL ASSIGNMENT


---

> This notebook is for ADALL ASSIGNMENT


---



## Introduction and Content

This dataset contains 7,000 synthetic transaction records designed to bridge the gap between "clean" classroom data and the messy reality of production environments. It provides a mix of 12 numerical and categorical features that simulate typical banking and e-commerce signals, like transaction velocity, customer age, and geographic distance from home

## 📖 Data Dictionary

| Column | Description |
|--------|-------------|
| `person_age` | Age of the person |
| `person_gender` | Gender of the person	Categorical|
| `person_education` | Highest education level |
| `person_income` | Annual income |
| `person_emp_exp` | Years of employment experience |
| `person_home_ownership` | Home ownership status (e.g., rent, own, mortgage) |
| `loan_amnt` | Loan amount requested |
| `loan_intent` | Purpose of the loan |
| `loan_int_rate` | Loan interest rate |
| `loan_percent_income` | Loan amount as a percentage of annual income |
| `cb_person_cred_hist_length` | Length of credit history in years |
| `credit_score` | Credit score of the person |
| `previous_loan_defaults_on_file` | Indicator of previous loan defaults |
| `loan_status (target variable)` | Loan approval status: 1 = approved; 0 = rejected |

---

URL = https://www.kaggle.com/datasets/taweilo/loan-approval-classification-data/data

EDA: https://www.kaggle.com/code/fatemehmohammadinia/personal-loan-acceptance-prediction-95-accuracy

This project focuses on building a classification model to predict the acceptance of personal loans by bank customers based on various demographic and financial features such as Age, Experience, Income, Family, Credit Card Average Spending (CCAvg), Education, Mortgage, and other account-related attributes including Securities Account, CD Account, Online Banking, and Credit Card Ownership.

---

Through this analysis, we aim to:

Explore and understand the structure, quality, and relationships within the dataset.
Perform data cleaning, feature transformation, and scaling to prepare the data for machine learning models.
Visualize key patterns, distributions, and correlations between features and loan acceptance behavior.
Build and evaluate classification models such as Logistic Regression, Naive Bayes, and K-Nearest Neighbors (KNN).
Interpret model outcomes to identify key factors influencing the likelihood of a customer accepting a personal loan.
This model is designed to assist banks and financial institutions in improving marketing strategies, customer targeting, and loan offer optimization by predicting which clients are most likely to accept a personal loan.

## What Is This Project About?
Personal loan acceptance depends on several demographic and financial factors such as income, education, family size, mortgage, credit card spending (CCAvg), and existing account relationships.
Accurate prediction of which customers are likely to accept a personal loan helps banks and financial institutions optimize their marketing campaigns and target the right customers more effectively.

---

In this project, we explore how machine learning classification models can be used to predict whether a customer will accept a personal loan based on their financial and demographic attributes.
The dataset is based on real-world customer banking data that includes behavioral and financial indicators.

##Why Predict Loan Acceptance?

Predicting loan acceptance is useful for:

Banks & financial institutions: Targeting customers with higher acceptance probability

Marketing teams: Designing cost-efficient and personalized loan offers

Risk analysts: Understanding which customers are less likely to respond

Decision-makers: Enhancing strategic planning for future campaigns
By applying classification models, we can transform customer data into actionable insights that improve business outcomes.

##Prompt example

You are helping a data analytics student prepare a laptop pricing dataset for modelling.

- Do not assume external knowledge.
- Do not say to drop a column just because it is listed as a warning.
- Keep the answer concise.
- Do not invent causes.
- Do not overclaim.

In [1]:
# Libraries used in this notebook
import os
import shutil
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Modelling libraries used later in Session 2
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ShuffleSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

RANDOM_STATE = 42

# Make wide tables easier to read in Colab
pd.set_option('display.max_columns', 100)

print('Setup complete')

Setup complete


In [2]:
github_raw_url = 'https://raw.githubusercontent.com/jiannyuhs-hue/ADALL_github/refs/heads/main/ADALL%20assignment/loan_data.csv'

df = pd.read_csv(github_raw_url)

print('Dataset loaded successfully.')
print('Shape:', df.shape)
display(df.head())

# This cell loads the laptop price dataset directly from GitHub.
# The GitHub raw URL points to the actual CSV file, not the normal GitHub preview page.
# pd.read_csv() reads the CSV file and stores it as a pandas DataFrame called df.
# df.shape shows the number of rows and columns in the dataset.
# df.head() displays the first 5 rows so we can quickly check that the data loaded correctly.

Dataset loaded successfully.
Shape: (45000, 14)


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [3]:
# Default method: copy the prompt into the chatbot manually.
RUN_API_CELLS = True
OPENAI_MODEL = 'gpt-5.4-nano'
client = None

# Only use this section if your tutor has asked you to call the API from Colab.
# You must first save OPENAI_API_KEY in Colab Secrets.

if RUN_API_CELLS == True:
    from google.colab import userdata
    from openai import OpenAI

    api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=api_key)
    print('OpenAI client is ready.')
else:
    print('Manual chatbot mode. Copy the prompts when they appear.')

# This cell prepares the notebook to use the OpenAI API, if needed.
# RUN_API_CELLS controls whether the notebook uses the API or manual chatbot mode.
# If RUN_API_CELLS is True, the notebook will try to connect to OpenAI using an API key.
# If RUN_API_CELLS is False, students can copy the prompt and paste it into ChatGPT manually.
# OPENAI_MODEL stores the model name that will be used later when sending prompts.
# client starts as None first, then becomes an OpenAI client after the API key is loaded.
# userdata.get('OPENAI_API_KEY') reads the API key saved in Colab Secrets.
# OpenAI(api_key=api_key) creates the connection object used to call the API.

OpenAI client is ready.


In [4]:
# Convert the first few rows to a string to send to OpenAI
data_preview = df.head(100).to_csv(index=False)
print(data_preview)

person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
21.0,female,High School,12951.0,0,OWN,2500.0,VENTURE,7.14,0.19,2.0,532,No,1
26.0,female,Bachelor,93471.0,1,RENT,35000.0,EDUCATION,12.42,0.37,3.0,701,No,1
24.0,female,High School,95550.0,5,RENT,35000.0,MEDICAL,11.11,0.37,4.0,585,No,1
24.0,female,Associate,100684.0,3,RENT,35000.0,PERSONAL,8.9,0.35,2.0,544,No,1
21.0,female,High School,12739.0,0,OWN,1600.0,VENTURE,14.74,0.13,3.0,640,N

In [5]:
prompt_dataset = f"""

You are an expert data scientist with experience in tree-based regression models.
Here are the first 100 rows of a dataset:\n{data_preview}\n\n

Dataset context: The dataset contains the acceptance of personal loans by bank customers with various demographic and financial features.

Please answer:

1) What should the  Business problem be?
2) What should the modelling objective be?
3) What is the most meaningful target column?
4) Which metric would be easiest to explain to business users?
5) Who are the main stakeholders?
6) What are three risks or pitfalls?

Keep the answer concise.
Do not invent causes.
Do not overclaim.

"""

response = client.responses.create(
    model="gpt-5-mini",
    input=prompt_dataset,
)

print(response.output_text)

1) Business problem
- Improve marketing/resource allocation by identifying which customers are most likely to accept a personal loan offer, so the bank can increase conversion while controlling outreach cost and credit risk.

2) Modelling objective
- Build a predictive model that estimates each customer’s probability of accepting a personal loan (a binary classification / probability scoring task) to rank/prioritize outreach and inform expected return per offer.

3) Most meaningful target column
- loan_status (binary 1/0) — use as the label representing whether the customer accepted the loan offer.

4) Metric easiest to explain to business users
- Conversion rate among targeted customers (Precision): “Of customers we predict will accept, what percent actually accept?” — intuitive for marketing ROI. (If business cares about overall ranking, complement with top-decile lift.)

5) Main stakeholders
- Marketing / Customer Acquisition (targets and campaign spend)
- Credit Risk / Underwriting